# 🛺 DEA Market Intelligence Agent (2025)
### Agentic AI + Tool Use + Local LLM (Ollama) | No API Key Required

**What this is:**  
An AI agent that answers business questions in plain English by querying real VAHAN data.  
Runs 100% locally using Ollama + Llama 3.1 — no API key, no cost, no internet required.

**Stack:** Python + Pandas + Ollama (llama3.1) + Tool Use

> ⚙️ **Setup (one time only):**
> ```bash
> ollama pull llama3.1
> ```
> Then just Run All — no other setup needed.

---


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import ollama
import json, warnings
warnings.filterwarnings('ignore')

MONTH_ORDER = ['JAN','FEB','MAR','APR','MAY','JUN',
               'JUL','AUG','SEP','OCT','NOV','DEC']

MODEL = 'llama3.1'  # change to 'llama3.2' or 'mistral' if preferred

df = pd.read_csv('/Users/admin/Desktop/TO DO /Project dea/data/market_clean.csv')
df['Month'] = pd.Categorical(df['Month'], categories=MONTH_ORDER, ordered=True)

# Verify Ollama is running
models = ollama.list()
available = [m.model for m in models.models]
print(f"✅ Data loaded: {len(df):,} rows")
print(f"✅ Ollama running | Available models: {available}")
print(f"✅ Using model: {MODEL}")


## 2. Define Agent Tools
Five Python functions that query real data. The agent decides which ones to call.


In [ ]:
def get_market_share(maker_group: str = None) -> dict:
    """Get market share for all groups or a specific one"""
    total = df['Registerations'].sum()
    share = (df.groupby('Maker_Group')['Registerations']
               .sum().sort_values(ascending=False).reset_index())
    share['market_share_pct'] = (share['Registerations'] / total * 100).round(2)
    if maker_group:
        row = share[share['Maker_Group'].str.lower() == maker_group.lower()]
        if len(row) == 0:
            return {"error": f"'{maker_group}' not found"}
        return row.to_dict('records')[0]
    return share.to_dict('records')


def get_state_opportunities(min_market_size: int = 0, max_dea_share: float = 100) -> dict:
    """Find states where market is large but DEA share is low"""
    state_total = df.groupby('State')['Registerations'].sum()
    dea_state   = (df[df['Maker_Group']=='Dilli Electric Auto']
                   .groupby('State')['Registerations'].sum())
    dea_share   = (dea_state / state_total * 100).round(2)

    result = pd.DataFrame({
        'state'        : state_total.index,
        'market_size'  : state_total.values,
        'dea_units'    : [int(dea_state.get(s, 0)) for s in state_total.index],
        'dea_share_pct': [float(dea_share.get(s, 0)) for s in state_total.index]
    })
    result = result[
        (result['market_size'] >= min_market_size) &
        (result['dea_share_pct'] <= max_dea_share)
    ].sort_values('market_size', ascending=False)
    return result.to_dict('records')


def compare_competitors(makers: list = None) -> dict:
    """Compare top competitors by registrations"""
    if makers:
        data = df[df['Maker'].isin(makers)]
    else:
        top  = df.groupby('Maker')['Registerations'].sum().sort_values(ascending=False).head(10)
        data = df[df['Maker'].isin(top.index)]
    result = (data.groupby('Maker')['Registerations']
                  .sum().sort_values(ascending=False).reset_index())
    total  = df['Registerations'].sum()
    result['market_share_pct'] = (result['Registerations'] / total * 100).round(2)
    return result.to_dict('records')


def get_monthly_trend(maker_group: str = 'Dilli Electric Auto') -> dict:
    """Get monthly trend + MoM growth for a maker group"""
    data = df[df['Maker_Group'] == maker_group]
    if len(data) == 0:
        return {"error": f"No data for '{maker_group}'"}
    monthly   = data.groupby('Month')['Registerations'].sum().reindex(MONTH_ORDER)
    mom       = monthly.pct_change().mul(100).round(1)
    q4        = int(monthly[['OCT','NOV','DEC']].sum())
    q1        = int(monthly[['JAN','FEB','MAR']].sum())
    return {
        "maker_group"   : maker_group,
        "monthly_units" : {str(k): int(v) for k, v in monthly.items()},
        "mom_growth_pct": {str(k): float(v) for k, v in mom.dropna().items()},
        "best_month"    : str(monthly.idxmax()),
        "worst_month"   : str(monthly.idxmin()),
        "q4_units"      : q4, "q1_units": q1,
        "q4_vs_q1_ratio": round(q4/q1, 2) if q1 > 0 else None,
        "total_annual"  : int(monthly.sum())
    }


def get_dea_state_performance(state: str = None) -> dict:
    """Get DEA performance by state"""
    dea      = df[df['Maker_Group'] == 'Dilli Electric Auto']
    state_mkt= df.groupby('State')['Registerations'].sum()
    if state:
        dea_s   = dea[dea['State'] == state]
        if len(dea_s) == 0:
            return {"error": f"No DEA data for '{state}'"}
        monthly = dea_s.groupby('Month')['Registerations'].sum().reindex(MONTH_ORDER)
        mkt_sz  = int(state_mkt.get(state, 0))
        dea_tot = int(monthly.sum())
        return {
            "state": state, "dea_total": dea_tot, "market_size": mkt_sz,
            "dea_share_pct": round(dea_tot/mkt_sz*100, 2) if mkt_sz > 0 else 0,
            "best_month": str(monthly.idxmax()),
            "monthly_units": {str(k): int(v) for k, v in monthly.items()}
        }
    result = []
    for s in dea['State'].unique():
        dea_s = dea[dea['State']==s]['Registerations'].sum()
        mkt_s = state_mkt.get(s, 0)
        result.append({
            "state": s, "dea_units": int(dea_s), "market_size": int(mkt_s),
            "dea_share_pct": round(dea_s/mkt_s*100, 2) if mkt_s > 0 else 0
        })
    return sorted(result, key=lambda x: x['dea_units'], reverse=True)


# Tool executor
TOOL_MAP = {
    "get_market_share"          : get_market_share,
    "get_state_opportunities"   : get_state_opportunities,
    "compare_competitors"       : compare_competitors,
    "get_monthly_trend"         : get_monthly_trend,
    "get_dea_state_performance" : get_dea_state_performance,
}

def execute_tool(name, args):
    if name not in TOOL_MAP:
        return {"error": f"Unknown tool: {name}"}
    return TOOL_MAP[name](**args)

print("✅ 5 tools ready")


## 3. Tool Schema for Ollama

In [ ]:
# Ollama uses the same OpenAI-style tool schema
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_market_share",
            "description": "Get market share breakdown for EV 3-wheeler brand groups in India 2025. Use for questions about who dominates the market or DEA's overall share.",
            "parameters": {
                "type": "object",
                "properties": {
                    "maker_group": {"type": "string", "description": "Optional. Specific brand group e.g. 'Dilli Electric Auto'. Leave empty for all."}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_state_opportunities",
            "description": "Find states where EV market is large but DEA share is low — expansion opportunities. Use for questions about which state to enter or target next.",
            "parameters": {
                "type": "object",
                "properties": {
                    "min_market_size": {"type": "integer", "description": "Minimum state market size. Default 0."},
                    "max_dea_share"  : {"type": "number",  "description": "Max DEA share % to filter low-penetration states. Default 100."}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "compare_competitors",
            "description": "Compare top EV competitors by sales and market share. Use for questions about rivals, threats, or competitive position.",
            "parameters": {
                "type": "object",
                "properties": {
                    "makers": {"type": "array", "items": {"type": "string"}, "description": "Optional list of maker names. Leave empty for top 10."}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_monthly_trend",
            "description": "Get monthly registration trend and MoM growth for a brand. Use for questions about seasonality, best/worst months, or growth trends.",
            "parameters": {
                "type": "object",
                "properties": {
                    "maker_group": {"type": "string", "description": "Brand group name. Default: 'Dilli Electric Auto'"}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_dea_state_performance",
            "description": "Get DEA's registrations broken down by state. Use for questions about DEA's best/worst states or specific state performance.",
            "parameters": {
                "type": "object",
                "properties": {
                    "state": {"type": "string", "description": "Optional. State name e.g. 'UP', 'Bihar', 'Delhi'. Leave empty for all states."}
                }
            }
        }
    }
]

print(f"✅ {len(TOOLS)} tools registered")


## 4. The Agent Loop

In [ ]:
SYSTEM_PROMPT = """You are a senior market intelligence analyst for Dilli Electric Auto (DEA),
an e-rickshaw and loader manufacturer in India.

You have tools that query real 2025 VAHAN registration data for EV 3-wheelers across 11 Indian states.
Always use a tool to get real data before answering. Never guess numbers.
After getting results, give a clear direct business answer using the actual numbers.
Keep answers concise and insightful — like a real analyst briefing."""


def run_agent(question: str, verbose: bool = True) -> str:
    """
    DEA Market Intelligence Agent.
    Accepts a plain English business question.
    Returns a data-backed written answer.
    """
    messages = [
        {"role": "system",  "content": SYSTEM_PROMPT},
        {"role": "user",    "content": question}
    ]

    if verbose:
        print(f"❓ Question: {question}")
        print("-" * 55)

    # Agent loop
    while True:
        response = ollama.chat(
            model   = MODEL,
            messages= messages,
            tools   = TOOLS
        )

        msg = response.message

        # Check for tool calls
        if msg.tool_calls:
            # Add assistant message to history
            messages.append({"role": "assistant", "content": msg.content or "", "tool_calls": msg.tool_calls})

            # Execute each tool
            for tool_call in msg.tool_calls:
                name = tool_call.function.name
                args = dict(tool_call.function.arguments) if tool_call.function.arguments else {}

                if verbose:
                    print(f"🔧 Agent calling: {name}({json.dumps(args)})")

                result = execute_tool(name, args)

                # Add tool result to history
                messages.append({
                    "role"   : "tool",
                    "content": json.dumps(result)
                })
        else:
            # No tool calls — final answer
            answer = msg.content
            if verbose:
                print(f"\n📊 Agent Answer:\n{answer}")
            return answer


print("✅ Agent ready — run the demo cells below")


## 5. Ask the Agent

### Q1: Which state should DEA expand to next?

In [ ]:
_ = run_agent("Which state should DEA expand to next and why?")


### Q2: Who is DEA's biggest threat?

In [ ]:
_ = run_agent("Who is DEA's biggest competitive threat and how far behind are we?")


### Q3: When to push inventory?

In [ ]:
_ = run_agent("When should DEA push maximum inventory based on seasonal trends?")


### Q4: Complex multi-tool question

In [ ]:
_ = run_agent(
    "Compare DEA's performance in UP vs Bihar — which has more growth potential?"
)


### Q5: Ask anything

In [ ]:
# Change this to any business question
your_question = "What is DEA's market share and how does it compare to YC Electric?"

_ = run_agent(your_question)


## 6. Architecture & Resume Talking Points

```
User Question (plain English)
         ↓
   Ollama (llama3.1) — running locally
   + System Prompt (analyst role)
   + Tool Definitions (5 tools)
         ↓
   Model decides: which tool(s) to call?
         ↓
   Tool executes on real Pandas DataFrame
         ↓
   Result sent back to model as context
         ↓
   Model writes final answer with real numbers
         ↓
   (Loop repeats if model needs more data)
```



---
*Built by Aarav | Stack: Python + Pandas + Ollama (llama3.1) | Project: Dilli Electric Auto*
